# pycheckers rule inspection

Visual inspection notebook for board-state representation and American checkers rules. Each rule case renders move rows with metadata, an arrowed move diagram, and before/after boards. This notebook intentionally avoids tablebase generation.

In [ ]:
%matplotlib inline

from pycheckers import (
    BoardState,
    moves_df,
    show_move_rows,
    show_turn_rows,
    square_mask,
    turns_df,
)
from pycheckers.bitboard import generate_move_templates

MOVE_COLUMNS = [
    "side",
    "from_r", "from_c", "to_r", "to_c", "dr", "dc",
    "is_capture", "captured_r", "captured_c", "promotes", "is_king",
    "from_mask_hex", "to_mask_hex", "captured_mask_hex",
    "before_black_hex", "before_white_hex", "before_kings_hex",
    "after_move_black_hex", "after_move_white_hex", "after_move_kings_hex",
]

def move_table(state, moves=None):
    return moves_df(state, moves)[MOVE_COLUMNS]

def turn_table(state, turn):
    return turns_df(state, [turn])[MOVE_COLUMNS]

## Legend

In each move diagram: blue marks the moving piece and arrow, green marks the destination, and red marks a captured piece. The metadata pane includes row/column coordinates plus hex encodings for primitive move masks and before/after board masks.

In [ ]:
templates = generate_move_templates()
{
    "primitive_templates": len(templates),
    "quiet_templates": sum(not move["is_capture"] for move in templates),
    "capture_templates": sum(move["is_capture"] for move in templates),
}

## Rule: Initial Quiet Moves

From the starting position, black has seven legal non-capturing moves.

In [ ]:
initial = BoardState.initial()
opening_moves = initial.legal_moves()
assert len(opening_moves) == 7
assert not any(move["is_capture"] for move in opening_moves)
fig, axes = show_move_rows(initial, opening_moves, title="initial legal moves")
move_table(initial, opening_moves)

## Rule: Forced Capture

If at least one capture is legal, quiet moves are excluded from `legal_moves`. This position also contains another black man that would otherwise have quiet moves.

In [ ]:
forced_capture = BoardState(
    square_mask(2, 1) | square_mask(2, 5),
    square_mask(3, 2),
    0,
    "B",
)
forced_moves = forced_capture.legal_moves()
assert len(forced_moves) == 1
assert all(move["is_capture"] for move in forced_moves)
fig, axes = show_move_rows(forced_capture, forced_moves, title="forced capture suppresses quiet moves")
move_table(forced_capture, forced_moves)

## Rule: A Capture Removes The Captured Piece

The before/after masks show the captured white piece removed and the black piece moved to the landing square.

In [ ]:
single_capture = BoardState(
    square_mask(2, 1),
    square_mask(3, 2),
    0,
    "B",
)
capture_move = single_capture.legal_moves()[0]
after_capture = single_capture.apply_move(capture_move)
assert after_capture.black == square_mask(4, 3)
assert after_capture.white == 0
fig, axes = show_move_rows(single_capture, [capture_move], title="single capture before and after")
move_table(single_capture, [capture_move])

## Rule: Multi-Jump Turn

A legal turn can contain multiple primitive captures. The side to move changes only after the full turn is complete.

In [ ]:
multi_jump = BoardState(
    square_mask(2, 1),
    square_mask(3, 2) | square_mask(5, 4),
    0,
    "B",
)
jump_turn = multi_jump.legal_turns()[0]
assert len(jump_turn) == 2
assert [state.side for state in multi_jump.turn_states(jump_turn)] == ["B", "B", "W"]
fig, axes = show_turn_rows(multi_jump, jump_turn, title="multi-jump turn")
turn_table(multi_jump, jump_turn)

## Rule: Quiet Promotion

A man that reaches the far row is crowned. Promotion is visible in the destination state's `kings` mask.

In [ ]:
quiet_promotion = BoardState(
    square_mask(6, 1),
    square_mask(0, 1),
    0,
    "B",
)
promotion_moves = quiet_promotion.legal_moves()
assert len(promotion_moves) == 2
assert move_table(quiet_promotion, promotion_moves)["promotes"].all()
fig, axes = show_move_rows(quiet_promotion, promotion_moves, title="quiet promotion choices")
move_table(quiet_promotion, promotion_moves)

## Rule: Promotion During Capture Stops The Chain

Under American checkers rules, a man that crowns during a jump stops there instead of continuing as a king on the same turn.

In [ ]:
capture_promotion = BoardState(
    square_mask(5, 0),
    square_mask(6, 1) | square_mask(6, 3),
    0,
    "B",
)
crowning_turn = capture_promotion.legal_turns()[0]
assert len(crowning_turn) == 1
final_crown = capture_promotion.apply_turn(crowning_turn)
assert final_crown.kings == square_mask(7, 2)
assert final_crown.white == square_mask(6, 3)
fig, axes = show_turn_rows(capture_promotion, crowning_turn, title="capture promotion stops chain")
turn_table(capture_promotion, crowning_turn)

## Rule: Men Move Forward Only

A black man on row 3 moves toward larger row numbers; backward quiet moves are not legal.

In [ ]:
black_man = BoardState(square_mask(3, 2), 0, 0, "B")
man_moves = black_man.legal_moves()
assert all(move["dr"] > 0 for move in man_moves)
fig, axes = show_move_rows(black_man, man_moves, title="black man forward moves")
move_table(black_man, man_moves)

## Rule: White Moves In The Opposite Direction

A white man moves toward smaller row numbers.

In [ ]:
white_man = BoardState(0, square_mask(4, 3), 0, "W")
white_moves = white_man.legal_moves()
assert all(move["dr"] < 0 for move in white_moves)
fig, axes = show_move_rows(white_man, white_moves, title="white man forward moves")
move_table(white_man, white_moves)

## Rule: Kings Move Backward And Forward

A king can move in all four diagonal directions when unobstructed.

In [ ]:
black_king = BoardState(square_mask(3, 2), 0, square_mask(3, 2), "B")
king_moves = black_king.legal_moves()
assert any(move["dr"] < 0 for move in king_moves)
assert any(move["dr"] > 0 for move in king_moves)
fig, axes = show_move_rows(black_king, king_moves, title="king quiet moves")
move_table(black_king, king_moves)

## Rule: Kings Capture Backward

A king can also capture backward; this is illegal for an uncrowned man in the same square.

In [ ]:
king_capture = BoardState(
    square_mask(4, 3),
    square_mask(3, 2),
    square_mask(4, 3),
    "B",
)
king_capture_moves = king_capture.legal_moves()
assert len(king_capture_moves) == 1
assert king_capture_moves[0]["is_capture"]
assert king_capture_moves[0]["dr"] < 0
fig, axes = show_move_rows(king_capture, king_capture_moves, title="king backward capture")
move_table(king_capture, king_capture_moves)

## Representation Validation

`BoardState` validates overlap, dark-square occupancy, king subset, and side-to-move.

In [ ]:
invalid_cases = {
    "off_dark_square": lambda: BoardState(square_mask(0, 0), 0, 0, "B"),
    "overlap": lambda: BoardState(square_mask(2, 1), square_mask(2, 1), 0, "B"),
    "king_without_piece": lambda: BoardState(square_mask(2, 1), 0, square_mask(3, 2), "B"),
    "bad_side": lambda: BoardState(square_mask(2, 1), 0, 0, "red"),
}
errors = {}
for name, factory in invalid_cases.items():
    try:
        factory()
    except ValueError as exc:
        errors[name] = str(exc)
errors